# MedGemma 1.5 4B: imaging demo on public images

AI Agentic Enterprises: real model outputs for the flagship LinkedIn video.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**.
2. Accept the model terms at https://huggingface.co/google/medgemma-1.5-4b-it (while logged in to Hugging Face).
3. Create a Hugging Face *read* token. In Colab's left sidebar 🔑 **Secrets**, add it as `HF_TOKEN` and enable notebook access.

Then Runtime → Run all. At the end, `medgemma_demo_results.zip` downloads.

All images are CC0 / public domain from Wikimedia Commons. **Research demo only: not for clinical use.**

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -U transformers accelerate

In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(userdata.get('HF_TOKEN'))

In [ ]:
import torch, time
from transformers import pipeline
MODEL_ID = "google/medgemma-1.5-4b-it"
t0 = time.time()
pipe = pipeline("image-text-to-text", model=MODEL_ID, torch_dtype=torch.bfloat16, device="cuda")
print(f"loaded {MODEL_ID} in {time.time()-t0:.0f}s; GPU mem {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
UA = {"User-Agent": "medgemma-demo/0.1 (AI Agentic Enterprises; research demo)"}
def commons(name): return f"https://commons.wikimedia.org/wiki/Special:FilePath/{name.replace(' ', '_')}?width=1024"

CASES = [
  dict(id="cxr_normal", domain="Chest X-ray", file="Chest Xray PA 3-8-2010.png", license="CC0", source_label="Normal PA chest X-ray",
       role="You are an expert radiologist.",
       prompt="Describe this chest X-ray. Give findings and an impression, written as a draft report for a radiologist to review."),
  dict(id="cxr_pneumonia", domain="Chest X-ray", file="X-ray of lobar pneumonia.jpg", license="CC0", source_label="Lobar pneumonia",
       role="You are an expert radiologist.",
       prompt="Describe this chest X-ray. Give findings and an impression, written as a draft report for a radiologist to review."),
  dict(id="fundus_normal", domain="Retina", file="Fundus photograph of normal left eye.jpg", license="CC0", source_label="Normal left fundus",
       role="You are an expert ophthalmologist.",
       prompt="Describe this retinal fundus photograph. Note any signs of diabetic retinopathy or other abnormality, and state whether referral is suggested."),
  dict(id="fundus_retinopathy", domain="Retina", file="Fundus retinopathy EDA03.JPG", license="Public domain", source_label="Diabetic retinopathy (NEI)",
       role="You are an expert ophthalmologist.",
       prompt="Describe this retinal fundus photograph. Note any signs of diabetic retinopathy or other abnormality, and state whether referral is suggested."),
  dict(id="derm_melanoma", domain="Dermatology", file="Melanoma Asymmetry.jpg", license="Public domain", source_label="Melanoma (NCI)",
       role="You are an expert dermatologist.",
       prompt="Describe this skin lesion using the ABCDE criteria and give a short differential diagnosis."),
  dict(id="path_adenocarcinoma", domain="Pathology", file="Histopathology of adenocarcinoma.png", license="CC0", source_label="Adenocarcinoma (H&E)",
       role="You are an expert pathologist.",
       prompt="Describe this H&E-stained histopathology image and suggest the most likely diagnosis."),
]

In [ ]:
import requests, io, json, os
from PIL import Image
from IPython.display import display, Markdown
os.makedirs("out/images", exist_ok=True)
results = []
for c in CASES:
    img = Image.open(io.BytesIO(requests.get(commons(c["file"]), headers=UA, timeout=60).content)).convert("RGB")
    img.save(f"out/images/{c['id']}.png")
    messages = [
        {"role": "system", "content": [{"type": "text", "text": c["role"]}]},
        {"role": "user", "content": [{"type": "text", "text": c["prompt"]}, {"type": "image", "image": img}]},
    ]
    t = time.time()
    out = pipe(text=messages, max_new_tokens=500, do_sample=False)
    secs = time.time() - t
    answer = out[0]["generated_text"][-1]["content"]
    results.append({**c, "source_url": f"https://commons.wikimedia.org/wiki/File:{c['file'].replace(' ', '_')}",
                    "model": MODEL_ID, "gpu": torch.cuda.get_device_name(0), "seconds": round(secs, 1), "output": answer})
    display(Markdown(f"### {c['domain']}: {c['id']}  \n*source label: {c['source_label']}; {secs:.1f}s*"))
    display(img.resize((320, int(320 * img.height / img.width))))
    display(Markdown(answer))

In [ ]:
json.dump(results, open("out/results.json", "w"), indent=2)
!cd out && zip -qr ../medgemma_demo_results.zip .
from google.colab import files
files.download("medgemma_demo_results.zip")